# 06 — 226-Class Maximum Accuracy Training Notebook

Bu notebook, önceki denemelerden çıkarılan derslerle **226 class** için yeni modeli eğitir.

Ana kararlar:

- Class silme yok: 226 class yeniden denenir.
- Depth kullanılmaz: sadece color landmarks kullanılır.
- Preprocessing final 184 model ile aynı mantıktadır:
  - color-only hand landmarks
  - relative hand coordinates
  - finger angle features
  - z-score normalization
- Feature shape: `16 × 156`
- Output: `226 class`
- Maksimum doğruluk için:
  - Conv1D + BiLSTM + Temporal Attention
  - Label smoothing
  - Sqrt-balanced sample weighting
  - Aktif augmentation
  - Top-1 / Top-3 / Top-5 raporlama
  - Per-class accuracy ve confusion analizi
  - Demo asset export

Gerekli landmark dosyaları:
`/content/drive/MyDrive/sign-language-project/packed_landmarks/train_X.npy`
`/content/drive/MyDrive/sign-language-project/packed_landmarks/val_X.npy`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, time, gc, pickle, shutil, random
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import top_k_accuracy_score, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
BASE = Path("/content/drive/MyDrive/sign-language-project")
DATASET = BASE / "dataset"
PACKED_DIR = BASE / "packed_landmarks"
SIGN_CSV = DATASET / "SignList_ClassId_TR_EN.csv"

RUN_NAME = "226_final_pipeline_maxacc"

OUT_DIR = BASE / RUN_NAME
DEMO_DIR = BASE / "demo_assets_226_final_pipeline"
PREP_DIR = PACKED_DIR / "preprocessed_226_final_pipeline"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DEMO_DIR.mkdir(parents=True, exist_ok=True)
PREP_DIR.mkdir(parents=True, exist_ok=True)

SAVE_PATH = BASE / "best_model_226_final_pipeline.keras"
LOG_PATH = BASE / "training_log_226_final_pipeline.csv"

SEQ_LEN = 16
COLOR_DIM = 126
FINAL_FEAT_DIM = 156

BATCH_SIZE = 64
EPOCHS_STAGE1 = 90
PATIENCE_STAGE1 = 14

RUN_STAGE2_FINETUNE = True
EPOCHS_STAGE2 = 25
PATIENCE_STAGE2 = 8

FORCE_PREPROCESS = False

print("BASE:", BASE, BASE.exists())
print("PACKED_DIR:", PACKED_DIR, PACKED_DIR.exists())
print("SIGN_CSV:", SIGN_CSV, SIGN_CSV.exists())
print("OUT_DIR:", OUT_DIR)
print("SAVE_PATH:", SAVE_PATH)

In [ ]:
train_x_path = PACKED_DIR / "train_X.npy"
train_y_path = PACKED_DIR / "train_y.npy"
val_x_path = PACKED_DIR / "val_X.npy"
val_y_path = PACKED_DIR / "val_y.npy"

assert train_x_path.exists(), f"Bulunamadı: {train_x_path}"
assert train_y_path.exists(), f"Bulunamadı: {train_y_path}"
assert val_x_path.exists(), f"Bulunamadı: {val_x_path}"
assert val_y_path.exists(), f"Bulunamadı: {val_y_path}"

X_train_raw = np.load(train_x_path)
y_train_raw = np.load(train_y_path)
X_val_raw = np.load(val_x_path)
y_val_raw = np.load(val_y_path)

print("X_train_raw:", X_train_raw.shape)
print("y_train_raw:", y_train_raw.shape)
print("X_val_raw:", X_val_raw.shape)
print("y_val_raw:", y_val_raw.shape)

assert X_train_raw.shape[1] == SEQ_LEN
assert X_val_raw.shape[1] == SEQ_LEN
assert X_train_raw.shape[-1] >= COLOR_DIM
assert X_val_raw.shape[-1] >= COLOR_DIM

X_train_color = X_train_raw[:, :, :COLOR_DIM].astype(np.float32)
X_val_color = X_val_raw[:, :, :COLOR_DIM].astype(np.float32)

print("Color-only train:", X_train_color.shape)
print("Color-only val:", X_val_color.shape)

In [ ]:
def landmark_quality_df(X_color, y, split_name):
    left = X_color[:, :, :63]
    right = X_color[:, :, 63:]

    left_detected = np.any(np.abs(left) > 1e-8, axis=2)
    right_detected = np.any(np.abs(right) > 1e-8, axis=2)
    any_detected = left_detected | right_detected
    both_detected = left_detected & right_detected

    df = pd.DataFrame({
        "split": split_name,
        "label": y.astype(int),
        "detected_frames": any_detected.sum(axis=1),
        "detected_rate": any_detected.mean(axis=1),
        "left_frames": left_detected.sum(axis=1),
        "right_frames": right_detected.sum(axis=1),
        "both_hands_frames": both_detected.sum(axis=1),
        "both_hands_rate": both_detected.mean(axis=1),
        "all_zero": np.all(np.abs(X_color) <= 1e-8, axis=(1,2)),
    })
    return df

q_train = landmark_quality_df(X_train_color, y_train_raw, "train")
q_val = landmark_quality_df(X_val_color, y_val_raw, "val")
q_all = pd.concat([q_train, q_val], ignore_index=True)

class_quality = (
    q_all.groupby(["split", "label"])
    .agg(
        n=("label", "count"),
        all_zero_count=("all_zero", "sum"),
        mean_detected_frames=("detected_frames", "mean"),
        mean_detected_rate=("detected_rate", "mean"),
        mean_both_hands_frames=("both_hands_frames", "mean"),
        mean_both_hands_rate=("both_hands_rate", "mean"),
    )
    .reset_index()
)

class_quality_path = OUT_DIR / "class_landmark_quality_before_226_training.csv"
class_quality.to_csv(class_quality_path, index=False, encoding="utf-8-sig")

print("Class landmark quality saved:", class_quality_path)

print("\nEn düşük detected_rate olan train class'ları:")
display(
    class_quality[class_quality["split"] == "train"]
    .sort_values(["mean_detected_rate", "n"], ascending=[True, False])
    .head(20)
)

print("\nAll-zero sample sayısı:")
display(q_all.groupby("split")["all_zero"].agg(["count", "sum", "mean"]).reset_index())

In [ ]:
def normalize_hands_relative(X):
    X = X.astype(np.float32)
    result = X.copy()
    N, T, D = result.shape

    assert T == SEQ_LEN
    assert D == COLOR_DIM

    for start in [0, 63]:
        hand = result[:, :, start:start+63].reshape(N, T, 21, 3)
        wrist = hand[:, :, 0:1, :]
        hand_rel = hand - wrist

        scale_vec = hand_rel[:, :, 9, :]
        scale = np.linalg.norm(scale_vec, axis=-1, keepdims=True)[:, :, :, np.newaxis]
        scale = np.where(scale < 1e-6, 1.0, scale)

        result[:, :, start:start+63] = (hand_rel / scale).reshape(N, T, 63)

    return result


def compute_finger_angles(X):
    X = X.astype(np.float32)
    N, T, D = X.shape

    assert T == SEQ_LEN
    assert D == COLOR_DIM

    finger_chains = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16],
        [17, 18, 19, 20],
    ]

    angles_all = []

    for hand_start in [0, 63]:
        hand = X[:, :, hand_start:hand_start+63].reshape(N, T, 21, 3)
        hand_angles = np.zeros((N, T, 15), dtype=np.float32)
        idx = 0

        for chain in finger_chains:
            for i in range(len(chain) - 1):
                a = hand[:, :, (0 if i == 0 else chain[i-1]), :]
                b = hand[:, :, chain[i], :]
                c = hand[:, :, chain[i+1], :]

                v1 = a - b
                v2 = c - b

                denom = (
                    np.linalg.norm(v1, axis=-1) *
                    np.linalg.norm(v2, axis=-1) +
                    1e-8
                )

                cos_a = np.sum(v1 * v2, axis=-1) / denom
                hand_angles[:, :, idx] = np.arccos(np.clip(cos_a, -1, 1))
                idx += 1

        angles_all.append(hand_angles)

    return np.concatenate([X, np.concatenate(angles_all, axis=-1)], axis=-1)


def preprocess_color_to_156(X_color):
    X_rel = normalize_hands_relative(X_color)
    X_feat = compute_finger_angles(X_rel)
    X_feat = np.nan_to_num(X_feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    assert X_feat.shape[-1] == FINAL_FEAT_DIM
    return X_feat

print("Preprocessing functions ready.")

In [ ]:
X_TR_FEAT_PATH = PREP_DIR / "X_train_226_color_rel_angles.npy"
X_VL_FEAT_PATH = PREP_DIR / "X_val_226_color_rel_angles.npy"
Y_TR_PATH = PREP_DIR / "y_train_original.npy"
Y_VL_PATH = PREP_DIR / "y_val_original.npy"

cache_exists = (
    X_TR_FEAT_PATH.exists() and X_VL_FEAT_PATH.exists()
    and Y_TR_PATH.exists() and Y_VL_PATH.exists()
)

if cache_exists and not FORCE_PREPROCESS:
    print("Preprocessed cache bulundu. Yeniden hesaplanmadı.")
    X_train_feat = np.load(X_TR_FEAT_PATH).astype(np.float32)
    X_val_feat = np.load(X_VL_FEAT_PATH).astype(np.float32)
    y_train = np.load(Y_TR_PATH).astype(int)
    y_val = np.load(Y_VL_PATH).astype(int)
else:
    print("Preprocessing başlıyor...")
    t0 = time.time()

    X_train_feat = preprocess_color_to_156(X_train_color)
    X_val_feat = preprocess_color_to_156(X_val_color)
    y_train = y_train_raw.astype(int)
    y_val = y_val_raw.astype(int)

    np.save(X_TR_FEAT_PATH, X_train_feat)
    np.save(X_VL_FEAT_PATH, X_val_feat)
    np.save(Y_TR_PATH, y_train)
    np.save(Y_VL_PATH, y_val)

    print(f"Preprocessing tamamlandı: {(time.time()-t0)/60:.1f} dk")

print("X_train_feat:", X_train_feat.shape)
print("X_val_feat:", X_val_feat.shape)
print("Feature range:", float(np.min(X_train_feat)), float(np.max(X_train_feat)))

In [ ]:
nz_train = ~np.all(np.abs(X_train_feat) <= 1e-8, axis=(1, 2))
nz_val = ~np.all(np.abs(X_val_feat) <= 1e-8, axis=(1, 2))

X_tr = X_train_feat[nz_train]
y_tr_orig = y_train[nz_train]

X_vl = X_val_feat[nz_val]
y_vl_orig = y_val[nz_val]

all_classes = sorted(np.unique(np.concatenate([y_tr_orig, y_vl_orig])).astype(int).tolist())

le = LabelEncoder()
le.fit(all_classes)

y_tr = le.transform(y_tr_orig)
y_vl = le.transform(y_vl_orig)

NUM_CLASSES = len(le.classes_)

print("Train samples:", X_tr.shape)
print("Val samples:", X_vl.shape)
print("NUM_CLASSES:", NUM_CLASSES)
print("First classes:", le.classes_[:20])
print("Last classes:", le.classes_[-20:])

if NUM_CLASSES != 226:
    print("UYARI: NUM_CLASSES 226 değil.")
    print("Bazı class'lar train/val landmarklarında kalmamış olabilir.")
else:
    print("✅ 226 class tamam.")

train_counts = pd.Series(y_tr_orig).value_counts().sort_index()
val_counts = pd.Series(y_vl_orig).value_counts().sort_index()

class_counts = pd.DataFrame({
    "class_id": all_classes,
    "train_count": [int(train_counts.get(c, 0)) for c in all_classes],
    "val_count": [int(val_counts.get(c, 0)) for c in all_classes],
})
class_counts["total_count"] = class_counts["train_count"] + class_counts["val_count"]

class_counts_path = OUT_DIR / "class_counts_226.csv"
class_counts.to_csv(class_counts_path, index=False, encoding="utf-8-sig")

display(class_counts.sort_values("train_count").head(30))
print("Class counts saved:", class_counts_path)

np.save(DEMO_DIR / "label_encoder_classes.npy", le.classes_)

In [ ]:
feat_mean = X_tr.mean(axis=(0, 1))
feat_std = X_tr.std(axis=(0, 1))
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)

X_tr_n = (X_tr - feat_mean) / feat_std
X_vl_n = (X_vl - feat_mean) / feat_std

X_tr_n = np.nan_to_num(X_tr_n, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_vl_n = np.nan_to_num(X_vl_n, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

assert X_tr_n.shape[-1] == FINAL_FEAT_DIM
assert X_vl_n.shape[-1] == FINAL_FEAT_DIM
assert not np.isnan(X_tr_n).any()
assert not np.isnan(X_vl_n).any()

norm_stats = {
    "mean": feat_mean.astype(float).tolist(),
    "std": feat_std.astype(float).tolist(),
}

with open(DEMO_DIR / "norm_stats.json", "w", encoding="utf-8") as f:
    json.dump(norm_stats, f, indent=2)

with open(OUT_DIR / "norm_stats_226.json", "w", encoding="utf-8") as f:
    json.dump(norm_stats, f, indent=2)

print("X_tr_n:", X_tr_n.shape, "mean:", float(X_tr_n.mean()), "std:", float(X_tr_n.std()))
print("X_vl_n:", X_vl_n.shape, "mean:", float(X_vl_n.mean()), "std:", float(X_vl_n.std()))

In [ ]:
y_tr_oh = keras.utils.to_categorical(y_tr, NUM_CLASSES).astype(np.float32)
y_vl_oh = keras.utils.to_categorical(y_vl, NUM_CLASSES).astype(np.float32)

raw_cw = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_tr
)

cw = np.sqrt(raw_cw)
cw = cw / np.mean(cw)
cw = np.clip(cw, 0.50, 3.00).astype(np.float32)

sample_weight_tr = cw[y_tr].astype(np.float32)

cw_df = pd.DataFrame({
    "model_label": np.arange(NUM_CLASSES),
    "original_class_id": le.inverse_transform(np.arange(NUM_CLASSES)),
    "raw_class_weight": raw_cw,
    "sqrt_clipped_weight": cw,
    "train_count": np.bincount(y_tr, minlength=NUM_CLASSES),
    "val_count": np.bincount(y_vl, minlength=NUM_CLASSES),
})

cw_path = OUT_DIR / "class_weights_226.csv"
cw_df.to_csv(cw_path, index=False, encoding="utf-8-sig")

display(cw_df.sort_values("train_count").head(30))
print("Class weights saved:", cw_path)
print("sample_weight range:", float(sample_weight_tr.min()), float(sample_weight_tr.max()))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
_SEQ = SEQ_LEN
_FD = FINAL_FEAT_DIM

@tf.function
def augment(x, y, sw):
    if tf.random.uniform(()) < 0.55:
        x = x + tf.random.normal(tf.shape(x), stddev=0.018, dtype=x.dtype)

    if tf.random.uniform(()) < 0.45:
        x = x * tf.random.uniform((), 0.92, 1.08, dtype=x.dtype)

    if tf.random.uniform(()) < 0.35:
        mask_len = tf.random.uniform((), 1, 3, dtype=tf.int32)
        start = tf.random.uniform((), 0, _SEQ - mask_len + 1, dtype=tf.int32)
        mask = tf.concat([
            tf.ones([start, _FD], dtype=x.dtype),
            tf.zeros([mask_len, _FD], dtype=x.dtype),
            tf.ones([_SEQ - start - mask_len, _FD], dtype=x.dtype)
        ], axis=0)
        x = x * mask

    if tf.random.uniform(()) < 0.25:
        keep = tf.cast(tf.random.uniform([1, _FD]) > 0.025, x.dtype)
        x = x * keep

    return x, y, sw


def make_train_ds(X, y, sw, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.float32), y.astype(np.float32), sw.astype(np.float32))
    )
    ds = ds.shuffle(min(len(X), 20000), reshuffle_each_iteration=True)
    ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


def make_val_ds(X, y, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.float32), y.astype(np.float32))
    )
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


train_ds = make_train_ds(X_tr_n, y_tr_oh, sample_weight_tr)
val_ds = make_val_ds(X_vl_n, y_vl_oh)

print("train_ds / val_ds ready")

In [ ]:
@keras.utils.register_keras_serializable()
class ReduceSumAxis1(layers.Layer):
    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[2])


def temporal_attention_block(x, seq_len, attn_hidden=512):
    score = layers.Dense(attn_hidden, activation="tanh")(x)
    score = layers.Dense(1)(score)
    score = layers.Flatten()(score)
    weights = layers.Activation("softmax", name="attention_weights")(score)
    weights = layers.Reshape((seq_len, 1))(weights)
    context = layers.Multiply()([x, weights])
    context = ReduceSumAxis1()(context)
    return context


def build_226_max_model(seq_len=16, feat_dim=156, num_classes=226):
    inputs = keras.Input(shape=(seq_len, feat_dim), name="landmark_sequence")

    x = layers.GaussianNoise(0.01)(inputs)
    x = layers.LayerNormalization()(x)

    x = layers.Conv1D(256, kernel_size=3, padding="same", activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Conv1D(256, kernel_size=3, padding="same", activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.20)(x)

    attn = temporal_attention_block(x, seq_len, attn_hidden=512)
    avg = layers.GlobalAveragePooling1D()(x)
    mx = layers.GlobalMaxPooling1D()(x)

    x = layers.Concatenate()([attn, avg, mx])

    x = layers.Dense(768, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Dense(384, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="class_probabilities"
    )(x)

    return keras.Model(inputs, outputs, name="maxacc_226_color_landmark_model")


keras.backend.clear_session()
gc.collect()

model = build_226_max_model(SEQ_LEN, FINAL_FEAT_DIM, NUM_CLASSES)
model.summary()

In [ ]:
initial_lr = 8e-4

optimizer = keras.optimizers.AdamW(
    learning_rate=initial_lr,
    weight_decay=1e-4,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        keras.metrics.CategoricalAccuracy(name="accuracy"),
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3"),
        keras.metrics.TopKCategoricalAccuracy(k=5, name="top5"),
    ]
)

print("Model compiled.")

In [ ]:
callbacks_stage1 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(SAVE_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=PATIENCE_STAGE1,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.CSVLogger(str(LOG_PATH)),
    keras.callbacks.BackupAndRestore(
        backup_dir=str(OUT_DIR / "training_backup_stage1")
    ),
]

print("Callbacks ready.")

In [ ]:
print("=" * 60)
print("STAGE-1 TRAINING START")
print("=" * 60)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_stage1,
    verbose=1
)

with open(OUT_DIR / "history_stage1.pkl", "wb") as f:
    pickle.dump(history1.history, f)

print("Stage-1 done.")

In [ ]:
if RUN_STAGE2_FINETUNE:
    print("=" * 60)
    print("STAGE-2 FINE-TUNING START")
    print("=" * 60)

    custom_objects = {
        "ReduceSumAxis1": ReduceSumAxis1,
        "Custom>ReduceSumAxis1": ReduceSumAxis1,
    }
    model = keras.models.load_model(
        SAVE_PATH,
        compile=False,
        custom_objects=custom_objects,
        safe_mode=False
    )

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=1e-4,
            weight_decay=5e-5,
            clipnorm=1.0
        ),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.03),
        metrics=[
            keras.metrics.CategoricalAccuracy(name="accuracy"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3"),
            keras.metrics.TopKCategoricalAccuracy(k=5, name="top5"),
        ]
    )

    sw_stage2 = np.sqrt(sample_weight_tr)
    sw_stage2 = sw_stage2 / np.mean(sw_stage2)
    sw_stage2 = np.clip(sw_stage2, 0.75, 2.0).astype(np.float32)

    train_ds_stage2 = make_train_ds(X_tr_n, y_tr_oh, sw_stage2)

    callbacks_stage2 = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(SAVE_PATH),
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=PATIENCE_STAGE2,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
        keras.callbacks.CSVLogger(str(OUT_DIR / "training_log_226_stage2.csv")),
        keras.callbacks.BackupAndRestore(
            backup_dir=str(OUT_DIR / "training_backup_stage2")
        ),
    ]

    history2 = model.fit(
        train_ds_stage2,
        validation_data=val_ds,
        epochs=EPOCHS_STAGE2,
        callbacks=callbacks_stage2,
        verbose=1
    )

    with open(OUT_DIR / "history_stage2.pkl", "wb") as f:
        pickle.dump(history2.history, f)

    print("Stage-2 done.")
else:
    print("Stage-2 skipped.")

In [ ]:
custom_objects = {
    "ReduceSumAxis1": ReduceSumAxis1,
    "Custom>ReduceSumAxis1": ReduceSumAxis1,
}

best_model = keras.models.load_model(
    SAVE_PATH,
    compile=False,
    custom_objects=custom_objects,
    safe_mode=False
)

print("Loaded best model:", SAVE_PATH)
print("Input:", best_model.input_shape)
print("Output:", best_model.output_shape)

proba_vl = best_model.predict(X_vl_n, batch_size=BATCH_SIZE, verbose=1)
pred_vl = np.argmax(proba_vl, axis=1)

labels_all = np.arange(NUM_CLASSES)

top1 = top_k_accuracy_score(y_vl, proba_vl, k=1, labels=labels_all)
top3 = top_k_accuracy_score(y_vl, proba_vl, k=min(3, NUM_CLASSES), labels=labels_all)
top5 = top_k_accuracy_score(y_vl, proba_vl, k=min(5, NUM_CLASSES), labels=labels_all)

print("=" * 60)
print("226 FINAL PIPELINE VALIDATION RESULTS")
print("=" * 60)
print(f"Top-1: {top1*100:.2f}%")
print(f"Top-3: {top3*100:.2f}%")
print(f"Top-5: {top5*100:.2f}%")
print(f"Val samples: {len(y_vl)}")
print(f"Classes: {NUM_CLASSES}")
print("=" * 60)

metrics = {
    "model_file": str(SAVE_PATH),
    "num_classes": int(NUM_CLASSES),
    "classes_original_ids": [int(x) for x in le.classes_.tolist()],
    "val_samples": int(len(y_vl)),
    "top1_accuracy": float(top1),
    "top3_accuracy": float(top3),
    "top5_accuracy": float(top5),
    "preprocessing": "color_only + relative_coords + finger_angles + zscore",
    "feature_dim": int(FINAL_FEAT_DIM),
    "seq_len": int(SEQ_LEN),
    "depth_used": False,
}

with open(OUT_DIR / "validation_metrics_226.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

np.save(OUT_DIR / "val_pred_proba_226.npy", proba_vl)
np.save(OUT_DIR / "val_pred_label_226.npy", pred_vl)
np.save(OUT_DIR / "val_true_label_226.npy", y_vl)

print("Metrics saved:", OUT_DIR / "validation_metrics_226.json")

In [ ]:
val_df = pd.DataFrame({
    "model_label": y_vl,
    "pred_model_label": pred_vl,
})
val_df["correct"] = val_df["model_label"] == val_df["pred_model_label"]
val_df["original_class_id"] = le.inverse_transform(val_df["model_label"])
val_df["pred_original_class_id"] = le.inverse_transform(val_df["pred_model_label"])

per_class = (
    val_df.groupby(["model_label", "original_class_id"])
    .agg(
        n=("correct", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

if SIGN_CSV.exists():
    sign_df = pd.read_csv(SIGN_CSV)
    if "ClassId" in sign_df.columns:
        per_class = per_class.merge(
            sign_df[["ClassId", "TR", "EN"]],
            left_on="original_class_id",
            right_on="ClassId",
            how="left"
        )
        per_class = per_class.drop(columns=["ClassId"])

per_class["accuracy_%"] = per_class["accuracy"] * 100
per_class = per_class.sort_values(["accuracy", "n"], ascending=[True, False])

per_class_path = OUT_DIR / "per_class_accuracy_226_validation.csv"
per_class.to_csv(per_class_path, index=False, encoding="utf-8-sig")

print("Per-class accuracy saved:", per_class_path)
print("\nWorst 30 classes:")
display(per_class.head(30))

print("\nBest 30 classes:")
display(per_class.sort_values(["accuracy", "n"], ascending=[False, False]).head(30))

In [ ]:
wrong_df = val_df[val_df["correct"] == False].copy()

if len(wrong_df) == 0:
    print("No wrong predictions.")
else:
    confusions = (
        wrong_df.groupby(["original_class_id", "pred_original_class_id"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    if SIGN_CSV.exists() and "sign_df" in globals() and "ClassId" in sign_df.columns:
        name_map_tr = dict(zip(sign_df["ClassId"], sign_df.get("TR", "")))
        name_map_en = dict(zip(sign_df["ClassId"], sign_df.get("EN", "")))

        confusions["true_TR"] = confusions["original_class_id"].map(name_map_tr)
        confusions["true_EN"] = confusions["original_class_id"].map(name_map_en)
        confusions["pred_TR"] = confusions["pred_original_class_id"].map(name_map_tr)
        confusions["pred_EN"] = confusions["pred_original_class_id"].map(name_map_en)

    conf_path = OUT_DIR / "top_confusions_226_validation.csv"
    confusions.to_csv(conf_path, index=False, encoding="utf-8-sig")

    print("Confusions saved:", conf_path)
    display(confusions.head(50))

cm = confusion_matrix(y_vl, pred_vl, labels=np.arange(NUM_CLASSES))
np.save(OUT_DIR / "confusion_matrix_226_validation.npy", cm)
print("Confusion matrix saved:", OUT_DIR / "confusion_matrix_226_validation.npy")

In [ ]:
shutil.copy(SAVE_PATH, DEMO_DIR / "model.keras")

np.save(DEMO_DIR / "label_encoder_classes.npy", le.classes_)

label_map_demo = {}
if SIGN_CSV.exists():
    sign_df = pd.read_csv(SIGN_CSV)
else:
    sign_df = pd.DataFrame()

for new_label in range(NUM_CLASSES):
    orig = int(le.inverse_transform([new_label])[0])

    item = {"original_class_id": orig, "TR": "", "EN": ""}
    if len(sign_df) and "ClassId" in sign_df.columns:
        row = sign_df[sign_df["ClassId"] == orig]
        if len(row):
            item["TR"] = str(row.iloc[0].get("TR", ""))
            item["EN"] = str(row.iloc[0].get("EN", ""))

    label_map_demo[int(new_label)] = item

with open(DEMO_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map_demo, f, ensure_ascii=False, indent=2)

with open(DEMO_DIR / "demo_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "seq_len": int(SEQ_LEN),
        "feat_dim": int(FINAL_FEAT_DIM),
        "num_classes": int(NUM_CLASSES),
        "preprocessing": "color_only + relative_coords + finger_angles + zscore",
        "model_file": "model.keras",
        "norm_stats_file": "norm_stats.json",
        "label_map_file": "label_map.json",
        "confidence_threshold": 0.85,
        "top_k_display": 3,
        "depth_used": False,
        "recommended_ui_rule": "if confidence >= 0.85 show single prediction else show Top-3"
    }, f, indent=2, ensure_ascii=False)

print("Demo assets saved:", DEMO_DIR)
for p in sorted(DEMO_DIR.glob("*")):
    print(" -", p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

In [ ]:
print("=" * 70)
print("226-CLASS TRAINING COMPLETE")
print("=" * 70)
print("Best model:", SAVE_PATH)
print("Demo assets:", DEMO_DIR)
print("Validation metrics:", OUT_DIR / "validation_metrics_226.json")
print("Per-class validation:", OUT_DIR / "per_class_accuracy_226_validation.csv")
print("Confusions:", OUT_DIR / "top_confusions_226_validation.csv")
print()
print(f"Top-1: {top1*100:.2f}%")
print(f"Top-3: {top3*100:.2f}%")
print(f"Top-5: {top5*100:.2f}%")
print("=" * 70)

## Sonraki adım

Bu eğitim bittikten sonra gerçek karar için modeli test setinde de ölçmek gerekir.

226 model için test notebookunda şu yollar kullanılmalı:

```python
MODEL_PATH = BASE / "best_model_226_final_pipeline.keras"
DEMO_DIR = BASE / "demo_assets_226_final_pipeline"
```

Ayrıca 184 model test notebookundaki sınıf filtreleme mantığı 226 modelde kullanılmamalıdır; testteki tüm class'lar modele dahil edilmelidir.